In [3]:
###Document Structure

from langchain_core.documents import Document


In [5]:
doc= Document(
    page_content= "This is the main text content i am using to create RAG",
    metadata= {
        "source": "NBC2020p2.pdf",
        "pages":1,
        "author": "Canadian Commission Building and Fire Codes",
        "title": "National Building Code of Canada",
        "date_created": "2025-31-03"
    }
)


In [6]:
## Create a simple txt file
import os
os.makedirs("../data/text_files/", exist_ok=True)


In [7]:
sample_texts={
    "../data/text_files/building_code.txt": """The National Building Code of Canada 2020 (NBC), together with the National Plumbing
Code of Canada 2020 (NPC), the National Fire Code of Canada 2020 (NFC) and the National
Energy Code of Canada for Buildings 2020 (NECB), was developed by the Canadian
Commission on Building and Fire Codes (CCBFC) as an objective-based national model code
that can be adopted by provincial and territorial governments.
In Canada, provincial and territorial governments have the authority to enact legislation
that regulates building design and construction within their jurisdictions. This may involve
the adoption of the NBC without change or with modifications to suit local needs, and
the enactment of other laws and regulations regarding building design and construction,
including requirements for professional involvement.
The NBC is a model code in the sense that it helps promote consistency among provincial
and territorial building codes. Persons involved in the design or construction of a building
should consult the provincial or territorial jurisdiction concerned to find out which building
code is applicable.
This edition of the NBC succeeds the 2015 edition.
Development of the National Model Codes
GOVERNANCE CHANGE NOTE: The national code development system underwent a
governance change in November 2022 to support efforts to harmonize construction codes
in jurisdictions throughout Canada. The CCBFC, which had been in place since 1991, was
dissolved and replaced by a new governance model in which the Canadian Board for
Harmonized Construction Codes (CBHCC) is responsible for developing, approving and
maintaining the National Model Codes based on the strategic priorities set by the Canadian
Table for Harmonized Construction Codes Policy. The 2020 National Model Codes were
developed by the CCBFC. In this section, references to the CCBFC are written in the past
tense to reflect the change in governance.
The CCBFC, an independent committee established by the National Research Council of
Canada (NRC), was responsible for the content of the 2020 editions of the National Model
Codes. The CCBFC was made up of volunteers from across the country and from all facets of
the Codes-user community. Members of the CCBFC and its standing committees included
builders, engineers, skilled trade workers, architects, building owners, building operators, fire
and building officials, manufacturers, and representatives of general interests.
The CCBFC was advised on scope, policy and technical issues pertaining to the Codes by
the Provincial/Territorial Policy Advisory Committee on Codes (PTPACC), which was a
committee of senior representatives from provincial/territorial ministries responsible for
building, fire, plumbing and energy regulation in their jurisdictions. The PTPACC was
created by the provinces and territories, with provision of guidance to the CCBFC as one of its
main functions. Through the PTPACC, the provinces and territories were engaged in every
phase of the Codes development process.
Codes Canada staff within the Construction Research Centre at the NRC provided technical
and administrative support to the CCBFC and its standing committees, and coordinated the
provision of evidence-based research to inform Codes development. The NRC publishes the
National Model Codes and periodic revisions to the Codes to address pressing issues.""",

"../data/text_files/intro_sample.txt": """The NBC sets out technical provisions for the design and construction of new buildings. It
also applies to the alteration, change of use and demolition of existing buildings.
The NBC establishes requirements to address the following five objectives:
• safety
• health
• accessibility
• fire and structural protection of buildings
• environment"""
    
}
for file_path, content in sample_texts.items():
    with open(file_path, "w",encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")




Sample text files created successfully.


In [8]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/building_code.txt", encoding="utf-8")

document = loader.load()

print(document)



[Document(metadata={'source': '../data/text_files/building_code.txt'}, page_content='The National Building Code of Canada 2020 (NBC), together with the National Plumbing\nCode of Canada 2020 (NPC), the National Fire Code of Canada 2020 (NFC) and the National\nEnergy Code of Canada for Buildings 2020 (NECB), was developed by the Canadian\nCommission on Building and Fire Codes (CCBFC) as an objective-based national model code\nthat can be adopted by provincial and territorial governments.\nIn Canada, provincial and territorial governments have the authority to enact legislation\nthat regulates building design and construction within their jurisdictions. This may involve\nthe adoption of the NBC without change or with modifications to suit local needs, and\nthe enactment of other laws and regulations regarding building design and construction,\nincluding requirements for professional involvement.\nThe NBC is a model code in the sense that it helps promote consistency among provincial\nand

In [9]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader  
from langchain_community.document_loaders import PyPDFLoader  

##load all the pdf files from the directory
dir_Loader= DirectoryLoader(
    "../data/pdf/", 
    glob="**/*.pdf", ##Pattern to match the files
    loader_cls=PyPDFLoader, ##loader class 
    show_progress=False
     )

pdf_documents = dir_Loader.load()
pdf_documents


[Document(metadata={'producer': 'itext-paulo-155 (itextpdf.sf.net - lowagie.com)', 'creator': 'pdftk-java 3.3.3', 'creationdate': '2025-03-26T14:43:01-04:00', 'author': 'National Research Council Canada', 'keywords': 'Codes, Construction, Building, Building codes, Canada, NBC, NRCC-CONST-56435E, NRCCode, NRCCode2020', 'moddate': '2025-03-27T16:24:12-04:00', 'subject': 'National Building Code of Canada', 'title': 'National Building Code of Canada 2020, second printing', 'source': '../data/pdf/NBC2020p2.pdf', 'total_pages': 1536, 'page': 0, 'page_label': '1'}, page_content='National Building Code of Canada 2020 Volume 1\nBUILDING\n  National Building Code of Canada 2020  \nVolume 1\nCANADIAN COMMISSION ON \nBUILDING AND FIRE CODES'),
 Document(metadata={'producer': 'itext-paulo-155 (itextpdf.sf.net - lowagie.com)', 'creator': 'pdftk-java 3.3.3', 'creationdate': '2025-03-26T14:43:01-04:00', 'author': 'National Research Council Canada', 'keywords': 'Codes, Construction, Building, Building 

###Define EmbeddingManager

In [10]:
import spacy
import numpy as np
from typing import List

class EmbeddingManager:
    """Handles document embedding generation using spaCy"""

    def __init__(self, model_name: str = "en_core_web_md"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"Loading embedding model: {self.model_name}")
        self.model = spacy.load(self.model_name)
        print(
            f"Model loaded. Dim = {self.model.vocab.vectors_length}"
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        return np.vstack([self.model(text).vector for text in texts])

    def get_embedding_dimension(self) -> int:
        return self.model.vocab.vectors_length



###Initialize Embedding Manager

In [11]:
embedding_manager = EmbeddingManager()

Loading embedding model: en_core_web_md
Model loaded. Dim = 300


### chunking


In [12]:


def chunk_text(text: str, nlp, max_chars: int = 800):
    doc = nlp(text)
    chunks = []
    current = ""

    for sent in doc.sents:
        if len(current) + len(sent.text) <= max_chars:
            current += " " + sent.text
        else:
            chunks.append(current.strip())
            current = sent.text

    if current.strip():
        chunks.append(current.strip())

    return chunks

    




###Load PDFs and create chunk

In [27]:
import fitz
import glob
import os
import uuid

pdf_paths = glob.glob("../data/pdf/**/*.pdf", recursive=True)

all_chunks = []
metadata = []

for pdf_path in pdf_paths:
    doc = fitz.open(pdf_path)

    for page_idx, page in enumerate(doc):
        page_text = page.get_text().strip()
        if not page_text:
            continue

        page_chunks = chunk_text(
            page_text,
            embedding_manager.model,
            max_chars=800
        )

        for i, chunk in enumerate(page_chunks):
            chunk_id = str(uuid.uuid4())

            all_chunks.append(chunk)

            metadata.append({
                "id": chunk_id,
                "source": pdf_path,
                "page": page_idx,
                "chunk_index": i,
                "text": chunk   
            })




In [28]:
print(len(all_chunks))
print(all_chunks[0][:200])

7180
BUILDING
  National Building Code of Canada 2020 
 Volume 1
CANADIAN COMMISSION ON 
BUILDING AND FIRE CODES


In [29]:
embeddings = embedding_manager.generate_embeddings(all_chunks)
print(embeddings.shape)

(7180, 300)


###VectorStoreDB (HNSW wrapper)

In [31]:
import hnswlib
import numpy as np
import json
from typing import List, Dict, Any

class VectorStoreDB:
    def __init__(self, dim: int, space: str = "cosine"):
        self.dim = dim
        self.space = space
        self.index = hnswlib.Index(space=space, dim=dim)
        self.metadata: Dict[int, Dict[str, Any]] = {}
        self.initialized = False

    def init(self, max_elements: int, ef_construction: int = 200, M: int = 16):
        self.index.init_index(
            max_elements=max_elements,
            ef_construction=ef_construction,
            M=M
        )
        self.index.set_ef(50)
        self.initialized = True

    def add(self, embeddings: np.ndarray, metadatas: List[Dict[str, Any]]):
        if not self.initialized:
            raise RuntimeError("Index not initialized")

        start_id = len(self.metadata)
        ids = np.arange(start_id, start_id + len(embeddings))

        self.index.add_items(embeddings, ids)

        for i, meta in zip(ids, metadatas):
            self.metadata[int(i)] = meta

    def search(self, query_vector, k=5):
        labels, distances = self.index.knn_query(query_vector, k=k)
        results = []
        
        for idx, dist in zip(labels[0], distances[0]):
            meta = self.metadata[idx]
            results.append({
            "text": meta["text"],      # ← chunk text
            "metadata": meta,          # ← full metadata
            "score": 1 - dist          # cosine similarity
             })
             
        return results

    def save(self, index_path: str, metadata_path: str):
        self.index.save_index(index_path)
        with open(metadata_path, "w") as f:
            json.dump(self.metadata, f)

    def load(self, index_path: str, metadata_path: str, max_elements: int):
        self.index.load_index(index_path, max_elements=max_elements)
        with open(metadata_path, "r") as f:
            self.metadata = {int(k): v for k, v in json.load(f).items()}
        self.initialized = True


###Initialize Vector DB

In [32]:
embedding_dim = embedding_manager.get_embedding_dimension()

vector_db = VectorStoreDB(dim=embedding_dim)
vector_db.init(max_elements=len(embeddings))

vector_db.add(embeddings, metadata)

print(f"Stored {len(metadata)} chunks in HNSW")



Stored 7180 chunks in HNSW


###RAG Retrieval (Query to chunks)

In [33]:
def retrieve(query: str, k: int = 5):
    query_embedding = embedding_manager.generate_embeddings([query])[0]
    results = vector_db.search(query_embedding, k=k)
    return results

In [35]:
results = retrieve("What is this document about?", k=5)

for r in results:
    print("-" * 80)
    print("Score:", r["score"])
    print("Source:", r["metadata"]["source"])
    print("Page:", r["metadata"].get("page"))
    print("Chunk index:", r["metadata"].get("chunk_index"))

--------------------------------------------------------------------------------
Score: 0.97294426
Source: ../data/pdf/NBC2020p2.pdf
Page: 195
Chunk index: 1
--------------------------------------------------------------------------------
Score: 0.9687286
Source: ../data/pdf/NBC2020p2.pdf
Page: 798
Chunk index: 1
--------------------------------------------------------------------------------
Score: 0.9687286
Source: ../data/pdf/NBC2020p2.pdf
Page: 1527
Chunk index: 1
--------------------------------------------------------------------------------
Score: 0.9673937
Source: ../data/pdf/NBC2020p2.pdf
Page: 789
Chunk index: 3
--------------------------------------------------------------------------------
Score: 0.9673937
Source: ../data/pdf/NBC2020p2.pdf
Page: 1518
Chunk index: 3


###Persist VectorDB + Metadata

In [36]:
vector_db.save(
    index_path="hnsw_index.bin",
    metadata_path="metadata.json"
)


###Reload VectorDB + Metadata

In [21]:
vector_db = VectorStoreDB(dim=embedding_manager.get_embedding_dimension())
vector_db.load(
    index_path="hnsw_index.bin",
    metadata_path="metadata.json",
    max_elements=len(metadata)
)

print("Vector DB reloaded successfully")


Vector DB reloaded successfully


In [37]:
retrieve("Explain the main topic", k=5)


[{'text': 'Additional Information\nNumbering System\nA consistent numbering system has been used throughout the National Model Codes. The\nfirst number indicates the Part of the Code; the second, the Section in the Part; the third,\nx\nNational Building Code of Canada 2020 Volume 1',
  'metadata': {'id': '51ad7961-fe7d-4536-b8aa-5d69dc2bdfa5',
   'source': '../data/pdf/NBC2020p2.pdf',
   'page': 10,
   'chunk_index': 5,
   'text': 'Additional Information\nNumbering System\nA consistent numbering system has been used throughout the National Model Codes. The\nfirst number indicates the Part of the Code; the second, the Section in the Part; the third,\nx\nNational Building Code of Canada 2020 Volume 1'},
  'score': np.float32(0.8562397)},
 {'text': 'The limiting distance is measured along a line\nperpendicular to the wall surface from the point closest to the property line.\n 4. Establish the line in Table 9.10.15.4. from which the maximum permitted percentage area of glazed\nopenings wil

In [38]:
def retrieve_by_text(text: str, k: int = 5):
    """
    Retrieve top-k chunks from HNSW vector DB based on a text query.
    """
    # Generate embedding for the query
    query_embedding = embedding_manager.generate_embeddings([text])[0]
    
    # Search vector DB
    results = vector_db.search(query_embedding, k=k)
    
    return results

###Retrieve information based on "load bearing"

In [44]:
query = "loadbearing"  # short query instead of full definition
results = retrieve_by_text(query, k=5)  # fetch top 5 relevant chunks

for r in results:
    print("-"*80)
    print("Score:", r["score"])
    print("Source:", r["metadata"]["source"])
    print("Page:", r["metadata"].get("page"))
    print("Chunk index:", r["metadata"].get("chunk_index"))
    print("Text snippet:", r["metadata"].get("text", ""))

--------------------------------------------------------------------------------
Score: 0.17561376
Source: ../data/pdf/NBC2020p2.pdf
Page: 685
Chunk index: 1
Text snippet: Conseil national de recherches du Canada, 2025
Division B
Appendix C
Table C-3 (Continued)
Province and Location
Sa(0.2) for Seismic Design
in Part 9
Maple Creek
0.069
Meadow Lake
0.055
Melfort
0.055
Melville
0.069
Moose Jaw
0.096
Nipawin
0.054
North Battleford
0.056
Prince Albert
0.055
Qu'Appelle
0.090
Regina
0.101
Rosetown
0.059
Saskatoon
0.057
Scott
0.057
Strasbourg
0.074
Swift Current
0.070
Uranium City
0.053
Weyburn
0.186
Yorkton
0.063
Manitoba
Beausejour
0.056
Boissevain
0.059
Brandon
0.054
Churchill
0.053
Dauphin
0.055
Flin Flon
0.054
Gimli
0.055
Island Lake
0.054
Lac du Bonnet
0.056
Lynn Lake
0.053
Morden
0.053
Neepawa
0.054
Pine Falls
0.056
Portage la Prairie
0.054
Rivers
0.058
Sandilands
0.055
Selkirk
0.055
Split Lake
0.053
Steinbach
0.055
Swan River
0.055
The Pas
0.054
Thompson
0.053
Virden
0.064
Winnipeg


###RAG With Generation (summarizes "load bearing info")

In [45]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY")
)



###RAG Retrieval + GROQ Generation

In [ ]:

# Step 1: Retrieve relevant chunks from HNSW
retrieved_docs = retrieve_by_text("loadbearing in the building Code?", k=5)  # your HNSW retrieval

# Step 2: Prepare context
context_text = "\n\n".join([r["text"] for r in retrieved_docs])

for r in retrieved_docs:
    print("=" * 80)
    print(r["text"])

# Step 3: Build prompt
prompt = f"""
You are answering using a building code document.

Extract everywhere loadbearing from the context.

If the definition is present, return it clearly.
If not present, say "Definition not found in context."

Context:
{context_text}

Question: {query}
Answer:"""

# Step 4: Ask Groq
response = llm.invoke([{"role": "user", "content": prompt}])
print(response.content)


For this reason each system should be tested after installation to ensure that the design intent is met.
 The minimum pressure differential is not intended to apply to locations in stair shafts when doors in their
proximity are open to adjacent floor areas.
 A-3.2.6.2.(4)
Limiting Smoke Movement. Measures to prevent the migration of smoke from floor
areas below the lowest exit storey into upper storeys include the following.
1)
 An elevator hoistway that passes through the floor above the lowest exit storey should not penetrate
the floor of the storey immediately below the lowest exit storey, unless there is a vestibule between the shaft
and each floor area below the lowest exit storey that
National Building Code of Canada 2020 Volume 1
Division B
3-247
2)
Where the exterior wall of the uppermost storey is set back from the exterior
wall of the storey below, the roof and floor space supporting the setback wall shall be
sheathed with a wood-based material between the exterior wall of th